In [ ]:
import os, subprocess, sys, tempfile, urllib.request, shutil, importlib.util

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
# Default: DO NOT clone the whole repo. Set to '1' to allow cloning.
ALLOW_CLONE = os.environ.get('AADS_ALLOW_CLONE', '0') in ('1', 'true', 'True')

def _download_file(raw_base, rel_path, dest_root):
    dest_path = Path(dest_root) / rel_path
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    url = f'{raw_base}/{rel_path}'
    print(f'Downloading {url} -> {dest_path}')
    with urllib.request.urlopen(url) as r, open(dest_path, 'wb') as f:
        f.write(r.read())
    return dest_path

def _ensure_aads_repo_on_path():
    if ALLOW_CLONE:
        if not CLONE_TARGET.exists():
            print(f"Cloning repository to {CLONE_TARGET}...")
            subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
        if str(CLONE_TARGET) not in sys.path:
            sys.path.insert(0, str(CLONE_TARGET))
        print(f"✓ Repo cloned/accessed: {CLONE_TARGET}")
        return CLONE_TARGET

    # Fetch minimal helper files from GitHub raw (default behavior)
    print('Fetching minimal helper scripts from GitHub raw...')
    repo = REPO_URL.rstrip('.git').rstrip('/')
    if repo.startswith('https://github.com/'):
        raw_base = repo.replace('https://github.com/', 'https://raw.githubusercontent.com/') + '/main'
    else:
        raise RuntimeError('Unsupported REPO_URL format for raw fetch: ' + REPO_URL)

    temp_dir = Path(tempfile.mkdtemp(prefix='aads_min_'))
    print(f'Using temp dir: {temp_dir}')
    files_to_fetch = [
        'scripts/notebook_helpers/cell_script_runner.py',
        'scripts/colab_simple_adapter_smoke_ui.py',
    ]
    try:
        for rel in files_to_fetch:
            _download_file(raw_base, rel, temp_dir)
    except Exception as e:
        shutil.rmtree(temp_dir)
        raise RuntimeError(f'Failed to download {rel}: {e}') from e

    if str(temp_dir) not in sys.path:
        sys.path.insert(0, str(temp_dir))
    print(f"✓ Minimal helpers downloaded to {temp_dir}")
    return temp_dir

repo_root = _ensure_aads_repo_on_path()

# Import bootstrap helpers (fallback to loading by path if needed)
try:
    from scripts.notebook_helpers.cell_script_runner import run_cell_script
except Exception:
    spec = importlib.util.spec_from_file_location('cell_script_runner', str(Path(repo_root)/'scripts'/'notebook_helpers'/'cell_script_runner.py'))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    run_cell_script = module.run_cell_script

print(f"✓ sys.path[0]: {sys.path[0]}")
# Run the bootstrap cell script
run_cell_script('nb4_cell01_bootstrap_access.py', globals())

In [ ]:
import importlib
from scripts import colab_simple_adapter_smoke_ui

colab_simple_adapter_smoke_ui = importlib.reload(colab_simple_adapter_smoke_ui)
launch_simple_adapter_smoke_ui = colab_simple_adapter_smoke_ui.launch_simple_adapter_smoke_ui

print('[UI] Adapter secimi ve tek goruntu tahmini paneli aciliyor. Panel acildiktan sonra yeni tahminler panel icinden yapilir.')
launch_simple_adapter_smoke_ui(ROOT, show_all_adapters=True, show_mirror_adapters=True)
